# 01 — Data Understanding

### Movie Ticket Booking Analytics · synthetic dataset v2

**Goal of this notebook:** understand the raw file *before* changing a single value.
Every cleaning decision later must be justified by something we observe here.

We will check, in order:

1. Shape and columns
2. Missing values
3. Duplicate bookings
4. Dates and times
5. Text consistency (city spellings, case, whitespace)
6. Numeric ranges and impossible values
7. Revenue-formula consistency
8. Show-level consistency
9. Occupancy consistency
10. Customer consistency
11. A first look at customer segments
12. An issue register with a planned treatment for each problem

> ⚠️ **All data is synthetic.** It was generated by `src/00_generate_data.py`
> with `random.seed(42)`. Movie titles, multiplex names and customers are
> invented. Nothing here describes real box-office performance, real cinemas
> or real people.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# Work whether the notebook is launched from the project root or from notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "bookings_raw.csv").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw" / "bookings_raw.csv"

# phone is an identifier, not a number. Forcing it to text stops pandas from
# reading 8069729672 as 8.069730e+09 and losing the exact digits.
df = pd.read_csv(RAW, dtype={"phone": "string"})

print("File :", RAW)
print("Rows :", f"{len(df):,}")
print("Cols :", len(df.columns))

File : D:\chinmay\movie-ticket-booking-analytics\data\raw\bookings_raw.csv
Rows : 10,300
Cols : 31


## Step 1 — Shape, columns and first rows

`df.shape` tells us how many rows and columns we have. `df.head()` shows the
first records so we can see what a booking actually looks like.

A quick reminder of the grain: **one row = one booking**, and a booking can
contain more than one seat. `seats` is the number of tickets in that booking.

In [2]:
print("Shape (rows, columns):", df.shape)
print()
print("Column names:")
for i, column in enumerate(df.columns, start=1):
    print(f"  {i:2d}. {column}")

Shape (rows, columns): (10300, 31)

Column names:
   1. booking_id
   2. customer_id
   3. customer_name
   4. customer_email
   5. phone
   6. age
   7. age_group
   8. gender
   9. city
  10. theatre_name
  11. screen
  12. seats_capacity
  13. movie_title
  14. genre
  15. language
  16. rating
  17. format
  18. show_id
  19. show_date
  20. show_time
  21. ticket_type
  22. seats
  23. ticket_price
  24. convenience_fee
  25. promo_code
  26. discount_amount
  27. total_amount
  28. payment_method
  29. booking_channel
  30. booking_status
  31. booking_datetime


In [3]:
display(df.head(3))

,booking_id,customer_id,customer_name,customer_email,phone,age,age_group,gender,city,theatre_name,screen,seats_capacity,movie_title,genre,language,rating,format,show_id,show_date,show_time,ticket_type,seats,ticket_price,convenience_fee,promo_code,discount_amount,total_amount,payment_method,booking_channel,booking_status,booking_datetime
0,BK0000001,CUS03000,Aadhya Pillai,aadhya.pillai3000@example.com,7993925679,62,55+,Female,Chennai,Marina Movies - Velachery,Screen 3,170,Kadal Nilavu,Romance,Tamil,7.1,2D,SH000179,2025-10-02,15:30,Standard,2,230,33,NONE,0,493,UPI,Mobile App,Confirmed,2025-09-02 08:45:39
1,BK0000002,CUS01769,Suresh George,suresh.george1769@example.com,7207758708,49,45-54,Male,Bengaluru,Orion Cinemas - Koramangala,Screen 2,220,Kadal Nilavu,Romance,Tamil,7.1,3D,SH000103,2025-10-02,21:30,Standard,3,365,36,FIRSTBUY5,55,1076,UPI,Mobile App,Confirmed,2025-09-03 20:09:35
2,BK0000003,CUS02888,Myra Krishnan,myra.krishnan2888@example.com,9459624384,22,18-24,Female,Chennai,Marina Movies - Velachery,Screen 3,170,Kadal Nilavu,Romance,Tamil,7.1,2D,SH000179,2025-10-02,15:30,Standard,2,245,36,NONE,0,526,Debit Card,Kiosk,Confirmed,2025-09-03 22:16:21


### Data types at a glance

`df.dtypes` shows how pandas read each column. Dates are usually read as
text (`str`) until we parse them, and numbers can be read as text if even one
value is not numeric. We check this now because a wrong type silently breaks
calculations later.

In [4]:
dtype_table = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
display(dtype_table)

,dtype,non_null,example
booking_id,str,10300,BK0000001
customer_id,str,10300,CUS03000
customer_name,str,10300,Aadhya Pillai
customer_email,str,10238,aadhya.pillai3000@example.com
phone,string,10240,7993925679
age,int64,10300,62
age_group,str,10250,55+
gender,str,10300,Female
city,str,10300,Chennai
theatre_name,str,10300,Marina Movies - Velachery


> **Why `phone` is text:** pandas would otherwise read a 10-digit number as a
> float (`8069729672.0`) and display it in scientific notation. A phone number
> is an identifier — we never add, average or compare it numerically — so
> reading it as text is the correct choice. This is a type decision we make
> *before* analysis, not a cosmetic one.

The full column definitions, allowed values and business rules are in
`docs/data_dictionary.md`. The most important definitions for the rest of the
project are:

| Term | Meaning |
|---|---|
| **Booking** | One row / one transaction |
| **Ticket** | One seat; total tickets = `SUM(seats)` |
| **Revenue** | `SUM(total_amount)` for **Confirmed** bookings only |
| **Occupancy** | Confirmed seats in a show ÷ that screen's `seats_capacity` |
| **Customer type** | New = 1 booking, Returning = 2–3, Loyal = 4+ in the period |

## Step 2 — Missing values

**Why it matters:** a blank email is harmless for revenue analysis, but a blank
`age_group` or `promo_code` can break a grouping or hide a discount. We need to
know *which* fields are affected and *how many* rows, not just "there are some
nulls".

`df.isna()` marks blanks, and `.mean()` on that True/False column gives the
share missing. We multiply by 100 to read it as a percentage.

In [5]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": (df.isna().mean() * 100).round(2),
})
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)
display(missing)

print("Total rows with at least one blank:", int(df.isna().any(axis=1).sum()))

,missing,percent
customer_email,62,0.60
phone,60,0.58
promo_code,51,0.50
age_group,50,0.49


Total rows with at least one blank: 223


**What we see:** only four columns have blanks — `customer_email`, `phone`,
`age_group` and `promo_code`. Every other field is complete. The counts are
small (roughly 0.5% of rows each), so we can repair them rather than throw
rows away.

**Important:** an empty `promo_code` is not always "no promo". If the row still
has a `discount_amount`, the promo existed and was simply not recorded. We can
infer it from the discount rate (5%, 10%, 15% or 20%), which is why we do not
blindly fill every blank with `NONE`.

## Step 3 — Duplicate bookings

There are two different questions here:

1. **Exact duplicate rows** — every field is identical. Easy to spot with
   `df.duplicated()`.
2. **Duplicate `booking_id`** — the same booking reference appears more than
   once. This is the dangerous one, because the IDs may differ in a small
   detail (a space, a corrected email) and still represent the same sale.

If we leave duplicates in, every KPI is inflated: revenue, tickets, bookings,
and even occupancy (the same seats get counted twice for one show).

In [6]:
print("Rows in file                 :", f"{len(df):,}")
print("Distinct booking_id values   :", f"{df['booking_id'].nunique():,}")
print("Rows with a repeated id      :", int(df['booking_id'].duplicated().sum()))
print("Exact duplicate rows (all cols):", int(df.duplicated().sum()))

duplicate_ids = df[df["booking_id"].duplicated(keep=False)].sort_values("booking_id")
print("\nExample duplicate records:")
display(duplicate_ids.head(6))

Rows in file                 : 10,300
Distinct booking_id values   : 10,000
Rows with a repeated id      : 300
Exact duplicate rows (all cols): 300



Example duplicate records:


,booking_id,customer_id,customer_name,customer_email,phone,age,age_group,gender,city,theatre_name,screen,seats_capacity,movie_title,genre,language,rating,format,show_id,show_date,show_time,ticket_type,seats,ticket_price,convenience_fee,promo_code,discount_amount,total_amount,payment_method,booking_channel,booking_status,booking_datetime
27,BK0000028,CUS00879,Sai Rawat,sai.rawat879@example.com,8256529060,47,45-54,Other,Delhi,Regal Multiplex - Dwarka,Screen 1,310,Chandni Chowk Express,Comedy,Hindi,6.9,IMAX,SH000052,2025-10-05,21:30,Standard,6,625,38,NONE,0,3788,Debit Card,Website,Confirmed,2025-09-19 20:56:35
10155,BK0000028,CUS00879,Sai Rawat,sai.rawat879@example.com,8256529060,47,45-54,Other,Delhi,Regal Multiplex - Dwarka,Screen 1,310,Chandni Chowk Express,Comedy,Hindi,6.9,IMAX,SH000052,2025-10-05,21:30,Standard,6,625,38,NONE,0,3788,Debit Card,Website,Confirmed,2025-09-19 20:56:35
40,BK0000041,CUS00755,Lakshmi Reddy,lakshmi.reddy755@example.com,6864876998,31,25-34,Female,Mumbai,Skyline Multiplex - Andheri,Screen 4,170,Dilli Nights,Thriller,Hindi,8.1,2D,SH000004,2025-10-18,12:15,Standard,2,305,33,STUDENT15,92,551,UPI,Mobile App,Confirmed,2025-09-24 01:29:27
10152,BK0000041,CUS00755,Lakshmi Reddy,lakshmi.reddy755@example.com,6864876998,31,25-34,Female,Mumbai,Skyline Multiplex - Andheri,Screen 4,170,Dilli Nights,Thriller,Hindi,8.1,2D,SH000004,2025-10-18,12:15,Standard,2,305,33,STUDENT15,92,551,UPI,Mobile App,Confirmed,2025-09-24 01:29:27
217,BK0000218,CUS04155,Trisha Bhandari,trisha.bhandari4155@example.com,9939249623,28,25-34,Female,Ahmedabad,Heritage Cinema - Navrangpura,Screen 1,330,Rangmanch,Drama,Hindi,7.4,IMAX,SH000267,2025-10-04,12:15,Standard,1,440,23,NONE,0,463,Credit Card,Mobile App,Confirmed,2025-10-04 05:41:10
10291,BK0000218,CUS04155,Trisha Bhandari,trisha.bhandari4155@example.com,9939249623,28,25-34,Female,Ahmedabad,Heritage Cinema - Navrangpura,Screen 1,330,Rangmanch,Drama,Hindi,7.4,IMAX,SH000267,2025-10-04,12:15,Standard,1,440,23,NONE,0,463,Credit Card,Mobile App,Confirmed,2025-10-04 05:41:10


**What we see:** 10,300 rows but only 10,000 distinct booking IDs. The 300
extra rows are **exact copies** of existing bookings, not new sales.

**Treatment:** drop the repeated IDs and keep the first occurrence. That
returns us to 10,000 bookings. Because the copies are identical, keeping the
first or the last does not change any value.

**A useful teaching point:** the duplicate rows also distort occupancy. If a
show's seats are counted twice, its occupancy is overstated, and a nearly-full
show can even cross 100%. We will measure that in Step 9.

## Step 4 — Dates and times

Dates stored as text cannot be sorted, filtered by month, or compared
reliably. We convert `show_date` and `booking_datetime` with
`pd.to_datetime(..., errors="coerce")`. `errors="coerce"` turns impossible
values into `NaT` (pandas' "not a time") instead of crashing, so we can count
them.

We also check three separate things:

- Is `show_date` inside the intended period (2025-10-01 → 2026-09-30)?
- Is `show_time` one of the six valid slots?
- Was the booking made **before** the show started?

In [7]:
START = pd.Timestamp("2025-10-01")
END = pd.Timestamp("2026-09-30")
VALID_TIMES = {"09:00", "12:15", "15:30", "18:45", "21:30", "23:45"}

df["show_date_parsed"] = pd.to_datetime(df["show_date"], format="%Y-%m-%d", errors="coerce")
df["booking_dt_parsed"] = pd.to_datetime(
    df["booking_datetime"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)

print("Unparseable show_date        :", int(df["show_date_parsed"].isna().sum()))
print("Unparseable booking_datetime :", int(df["booking_dt_parsed"].isna().sum()))
print("show_date range              :", df["show_date_parsed"].min(), "->", df["show_date_parsed"].max())
print("booking_datetime range       :", df["booking_dt_parsed"].min(), "->", df["booking_dt_parsed"].max())

invalid_time = ~df["show_time"].isin(VALID_TIMES)
outside_period = (df["show_date_parsed"] < START) | (df["show_date_parsed"] > END)

print("\nInvalid show_time rows       :", int(invalid_time.sum()))
print("show_date outside period     :", int(outside_period.sum()))

print("\nInvalid show_time values:")
display(df.loc[invalid_time, "show_time"].value_counts().rename("rows"))

print("Out-of-period show_date values:")
display(df.loc[outside_period, "show_date"].value_counts().rename("rows"))

Unparseable show_date        : 0
Unparseable booking_datetime : 0
show_date range              : 2025-10-01 00:00:00 -> 2027-12-31 00:00:00
booking_datetime range       : 2025-09-02 08:45:39 -> 2026-09-29 15:12:00

Invalid show_time rows       : 31
show_date outside period     : 21

Invalid show_time values:


show_time
25:30    12
abc      12
99:00     7
Name: rows, dtype: int64

Out-of-period show_date values:


show_date
2027-12-31    21
Name: rows, dtype: int64

In [8]:
# A booking must happen before its show starts. Rows with an invalid show_time
# cannot be checked until the time is repaired, so we exclude them here.
valid_time = df["show_time"].isin(VALID_TIMES)
show_dt = df["show_date_parsed"] + pd.to_timedelta(df["show_time"] + ":00", errors="coerce")
booked_after_show = valid_time & (df["booking_dt_parsed"] > show_dt)

print("Bookings made after the show started (valid times only):", int(booked_after_show.sum()))
print("Rows that cannot be checked until show_time is repaired:", int((~valid_time).sum()))
print("Bookings made on the same calendar day as the show:",
      int((df["booking_dt_parsed"].dt.date == df["show_date_parsed"].dt.date).sum()))

Bookings made after the show started (valid times only): 0
Rows that cannot be checked until show_time is repaired: 31
Bookings made on the same calendar day as the show: 4154


**What we see:** 20 rows have a `show_date` of `2027-12-31`, which is outside
the one-year period, and 31 rows have impossible `show_time` values (`25:30`,
`99:00`, `abc`) — 30 injected defects plus one duplicate copy. No booking with
a **valid** time was made after its show started. The 31 invalid-time rows
cannot be checked until their time is repaired; we will repeat the check after
cleaning.

**Treatment:** both `show_date` and `show_time` are **show-level** attributes.
Other bookings for the same `show_id` still hold the correct values, so we can
repair the bad rows with the most common value for that show instead of
guessing or dropping them.

> Note: the check deliberately excludes invalid times. Using a fake `00:00`
> would create false "booked after the show" alarms, which is a good reminder
> to repair inputs before validating derived rules.

## Step 5 — Text consistency

A city can be written as `Bengaluru`, `Bangalore`, `Banglore`, `bengaluru`,
`BENGALURU` or `" Bengaluru "`. To a human these are the same place. To
`groupby`, they are six different cities.

This is one of the most common real-world data problems, and it silently
splits a city's revenue into several small buckets.

We will:

1. Count the distinct raw labels.
2. Strip leading/trailing spaces and apply title case.
3. Map known spelling variants to one canonical name.

In [9]:
raw_labels = df["city"].value_counts()
print("Distinct raw city labels:", raw_labels.size)
display(raw_labels)

Distinct raw city labels: 34


city
Mumbai         1665
Hyderabad      1657
Delhi          1641
Bengaluru      1280
Chennai        1218
Pune            998
Kolkata         792
Ahmedabad       770
Banglore         58
Bangalore        54
CHENNAI          13
chennai          12
DELHI            12
hyderabad        12
bengaluru        10
 Mumbai          10
MUMBAI           10
 Pune             9
mumbai            9
 Chennai          7
 Hyderabad        7
 Kolkata          6
KOLKATA           6
AHMEDABAD         6
delhi             5
 Delhi            5
ahmedabad         5
 Bengaluru        5
PUNE              4
 Ahmedabad        4
pune              4
kolkata           2
BENGALURU         2
HYDERABAD         2
Name: count, dtype: int64

In [10]:
CITY_MAP = {
    "Bangalore": "Bengaluru",
    "Banglore": "Bengaluru",
}

df["city_norm"] = df["city"].str.strip().str.title().replace(CITY_MAP)

print("Distinct labels after normalization:", df["city_norm"].nunique())
display(df["city_norm"].value_counts())

needs_strip = df["city"].astype("string") != df["city"].astype("string").str.strip()
needs_case = (
    df["city"].astype("string").str.strip()
    != df["city"].astype("string").str.strip().str.title()
)
print("\nRows with leading/trailing spaces:", int(needs_strip.sum()))
print("Rows not in title case            :", int(needs_case.sum()))
print("Rows changed by normalization     :", int((df["city"] != df["city_norm"]).sum()))

# Identifiers should also be checked as text, not numbers.
phone_valid = df["phone"].str.fullmatch(r"[6-9]\d{9}").fillna(False)
email_valid = df["customer_email"].str.contains("@", na=False)
print("Phones that are not 10 digits starting 6-9:",
      int((df["phone"].notna() & ~phone_valid).sum()))
print("Non-blank emails without '@':",
      int((df["customer_email"].notna() & ~email_valid).sum()))

Distinct labels after normalization: 8


city_norm
Mumbai       1694
Hyderabad    1678
Delhi        1663
Bengaluru    1409
Chennai      1250
Pune         1015
Kolkata       806
Ahmedabad     785
Name: count, dtype: int64


Rows with leading/trailing spaces: 53
Rows not in title case            : 114
Rows changed by normalization     : 279
Phones that are not 10 digits starting 6-9: 0
Non-blank emails without '@': 0


**What we see:** 34 raw labels collapse to the 8 real cities. The biggest
problem is Bengaluru: `Bengaluru`, `Bangalore` and `Banglore` are three
spellings of one city. In v1 of this dataset those spellings were generated as
three **separate, equally weighted** cities, which made Bengaluru look three
times larger than it really was. v2 fixes that at the source: Bengaluru has one
weight, and the variants are only labels.

**Treatment:** strip, title-case and map the variants. The canonical city then
matches the customer's home city and the theatre's city.

## Step 6 — Numeric ranges and impossible values

`describe()` gives the minimum, maximum and quartiles. A minimum age of `-5`
or a maximum rating of `12.5` is immediately suspicious. We check each field
against a **business rule**, not against a statistical outlier threshold:

| Field | Valid range | Why |
|---|---|---|
| `age` | 18–68 | The generator's customer pool |
| `seats` | 1–6 | A single booking cannot hold more |
| `ticket_price` | > 0 | A free/negative ticket is not a sale |
| `rating` | 0–10 | Standard rating scale |
| `convenience_fee` | ≥ 0 | A fee cannot be negative |
| `discount_amount` | ≥ 0 and ≤ subtotal | A discount cannot exceed the price |
| `total_amount` | > 0 | Money collected must be positive |
| `seats` vs `seats_capacity` | seats ≤ capacity | You cannot sell more seats than exist |

In [11]:
numeric_columns = [
    "age", "seats", "ticket_price", "convenience_fee",
    "discount_amount", "total_amount", "seats_capacity", "rating",
]
display(df[numeric_columns].describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
age,10300.0,35.87,12.75,-5.0,26.0,33.0,44.0,150.0
seats,10300.0,2.46,1.27,1.0,2.0,2.0,3.0,6.0
ticket_price,10300.0,475.45,221.28,-100.0,310.0,425.0,580.0,1500.0
convenience_fee,10300.0,32.54,7.47,20.0,26.0,33.0,39.0,45.0
discount_amount,10300.0,41.68,93.79,0.0,0.0,0.0,42.0,1315.0
total_amount,10300.0,1163.53,822.06,189.0,590.0,942.0,1484.0,8355.0
seats_capacity,10300.0,216.62,73.61,100.0,160.0,190.0,290.0,340.0
rating,10300.0,7.64,0.57,-1.0,7.2,7.5,8.1,12.5


In [12]:
subtotal = df["ticket_price"] * df["seats"]

checks = {
    "age outside 18-68": (df["age"] < 18) | (df["age"] > 68),
    "seats outside 1-6": (df["seats"] < 1) | (df["seats"] > 6),
    "ticket_price <= 0": df["ticket_price"] <= 0,
    "rating outside 0-10": (df["rating"] < 0) | (df["rating"] > 10),
    "convenience_fee < 0": df["convenience_fee"] < 0,
    "discount_amount < 0": df["discount_amount"] < 0,
    "discount > subtotal (valid prices)": (df["ticket_price"] > 0) & (df["discount_amount"] > subtotal),
    "total_amount <= 0": df["total_amount"] <= 0,
    "seats > seats_capacity": df["seats"] > df["seats_capacity"],
}

check_table = pd.DataFrame({
    "check": list(checks.keys()),
    "rows_failing": [int(mask.sum()) for mask in checks.values()],
})
display(check_table)

,check,rows_failing
0,age outside 18-68,11
1,seats outside 1-6,0
2,ticket_price <= 0,51
3,rating outside 0-10,10
4,convenience_fee < 0,0
5,discount_amount < 0,0
6,discount > subtotal (valid prices),0
7,total_amount <= 0,0
8,seats > seats_capacity,0


In [13]:
print("Rows with an impossible age:")
display(df.loc[checks["age outside 18-68"], ["booking_id", "customer_id", "age", "age_group"]].head(10))

print("Rows with an impossible rating:")
display(df.loc[checks["rating outside 0-10"], ["booking_id", "movie_title", "rating"]].head(10))

print("Rows with a non-positive ticket price:")
display(df.loc[checks["ticket_price <= 0"], ["booking_id", "ticket_price", "seats", "total_amount"]].head(10))

Rows with an impossible age:


,booking_id,customer_id,age,age_group
336,BK0000337,CUS02997,-5,25-34
2721,BK0002722,CUS03384,150,45-54
2826,BK0002827,CUS01284,-5,55+
5024,BK0005025,CUS02513,-5,18-24
5424,BK0005425,CUS04196,150,45-54
5481,BK0005482,CUS02339,-5,25-34
5665,BK0005666,CUS01786,150,35-44
6746,BK0006747,CUS00897,150,35-44
8524,BK0008525,CUS00095,-5,18-24
9391,BK0009392,CUS02326,150,18-24


Rows with an impossible rating:


,booking_id,movie_title,rating
80,BK0000081,Rangmanch,12.5
2468,BK0002469,Dilli Nights,-1.0
3664,BK0003665,Mehfil,-1.0
4002,BK0004003,Kadal Nilavu,12.5
4342,BK0004343,Dhoop Chhaon,12.5
4997,BK0004998,Aakhri Raasta,-1.0
5799,BK0005800,Iravu Payanam,-1.0
7214,BK0007215,Bhramaram,12.5
7419,BK0007420,War 2,-1.0
7915,BK0007916,Sivappu Manjal,12.5


Rows with a non-positive ticket price:


,booking_id,ticket_price,seats,total_amount
14,BK0000015,0,3,1324
545,BK0000546,-50,2,2255
1010,BK0001011,0,1,349
1014,BK0001015,-50,2,1495
1125,BK0001126,0,2,796
1230,BK0001231,-100,2,645
1258,BK0001259,-50,3,1490
1288,BK0001289,-50,3,1487
1347,BK0001348,-100,3,1574
1379,BK0001380,-100,5,4034


**What we see:** ages of `-5` and `150`, ratings of `-1.0` and `12.5`, and
ticket prices of `0`, `-50` and `-100`. Seats, capacity, fees and totals are
otherwise sensible: no booking sells more seats than the screen holds, and on
valid prices no discount exceeds the ticket subtotal. The discount check is
restricted to positive prices because a corrupted price of `0` or `-100` makes
the subtotal zero or negative — a symptom of the price defect, not a second
defect.

**Treatment:**

- **Age:** the same `customer_id` appears on other bookings with the correct
  age, so we copy the customer's modal age. If a customer had only one booking,
  we would fall back to the median age — but the defect was injected only into
  repeat customers, so this fallback is not needed.
- **Rating:** `rating` belongs to the movie, not the booking. We copy the modal
  rating for the same `movie_title`.
- **Ticket price:** `total_amount` was calculated from the original price, so
  we can recover it exactly:

  `ticket_price = (total_amount − convenience_fee + discount_amount) ÷ seats`

  This works because the formula is additive and all the other terms are intact.

These are **repairs**, not deletions. Dropping 70 rows would lose real
revenue and make the dataset smaller than the promised 10,000 bookings.

## Step 7 — Revenue-formula consistency

The data dictionary says:

`total_amount = ticket_price × seats + convenience_fee − discount_amount`

If this does not hold, the revenue KPI is wrong even if every individual
column looks plausible. We calculate the expected total ourselves and compare
it row by row.

In [14]:
expected_total = (
    df["ticket_price"] * df["seats"]
    + df["convenience_fee"]
    - df["discount_amount"]
)
mismatch = expected_total != df["total_amount"]

print("Rows where the formula fails:", int(mismatch.sum()))
print("Share of rows               :", f"{mismatch.mean() * 100:.2f}%")

comparison = df.loc[mismatch, [
    "booking_id", "ticket_price", "seats", "convenience_fee",
    "discount_amount", "total_amount",
]].copy()
comparison["expected"] = expected_total[mismatch]
display(comparison.head(10))

Rows where the formula fails: 51
Share of rows               : 0.50%


,booking_id,ticket_price,seats,convenience_fee,discount_amount,total_amount,expected
14,BK0000015,0,3,42,68,1324,-26
545,BK0000546,-50,2,23,118,2255,-195
1010,BK0001011,0,1,24,0,349,24
1014,BK0001015,-50,2,35,0,1495,-65
1125,BK0001126,0,2,40,134,796,-94
1230,BK0001231,-100,2,25,0,645,-175
1258,BK0001259,-50,3,35,0,1490,-115
1288,BK0001289,-50,3,33,256,1487,-373
1347,BK0001348,-100,3,44,270,1574,-526
1379,BK0001380,-100,5,29,445,4034,-916


**What we see:** 52 rows fail the formula, and every one of them has a
non-positive `ticket_price`. That is not a coincidence: the bad prices were
injected **without** changing `total_amount`, which is exactly what happens in
a real system when a price field is corrupted but the stored total is correct.

**Treatment:** recover the price from the formula, then recompute
`total_amount` and confirm that **0 rows** fail. This gives us a single,
auditable definition of revenue.

## Step 8 — Show-level consistency

`show_id` identifies one specific screening: one movie, in one screen, at one
theatre, on one date, at one time. Every booking for that show must agree on
those attributes.

This matters because occupancy is calculated **per show**. If the same
`show_id` had two different capacities or two different dates, the occupancy
denominator would be ambiguous.

In [15]:
show_check = df.groupby("show_id").agg(
    bookings=("booking_id", "count"),
    movies=("movie_title", "nunique"),
    theatres=("theatre_name", "nunique"),
    screens=("screen", "nunique"),
    capacities=("seats_capacity", "nunique"),
    dates=("show_date", "nunique"),
    times=("show_time", "nunique"),
)

print("Distinct shows:", len(show_check))
print("Shows with >1 movie/theatre/screen/capacity:",
      int((show_check[["movies", "theatres", "screens", "capacities"]] > 1).any(axis=1).sum()))
print("Shows with >1 show_date:", int((show_check["dates"] > 1).sum()))
print("Shows with >1 show_time:", int((show_check["times"] > 1).sum()))

display(show_check.head())

Distinct shows: 290
Shows with >1 movie/theatre/screen/capacity: 0
Shows with >1 show_date: 20
Shows with >1 show_time: 30


,bookings,movies,theatres,screens,capacities,dates,times
show_id,,,,,,,
SH000001,30,1,1,1,1,1,1
SH000002,13,1,1,1,1,1,1
SH000003,21,1,1,1,1,1,1
SH000004,16,1,1,1,1,1,2
SH000005,71,1,1,1,1,1,1


In [16]:
problem_shows = show_check[(show_check[["dates", "times"]] > 1).any(axis=1)]
display(problem_shows)

example_id = problem_shows.index[0]
display(df.loc[df["show_id"] == example_id, [
    "booking_id", "show_id", "movie_title", "theatre_name",
    "screen", "seats_capacity", "show_date", "show_time",
]].sort_values("show_date"))

,bookings,movies,theatres,screens,capacities,dates,times
show_id,,,,,,,
SH000004,16,1,1,1,1,1,2
SH000008,41,1,1,1,1,1,2
SH000009,51,1,1,1,1,2,1
SH000011,9,1,1,1,1,2,1
SH000026,46,1,1,1,1,1,2
SH000028,65,1,1,1,1,1,2
SH000033,18,1,1,1,1,1,2
SH000039,68,1,1,1,1,1,2
SH000040,12,1,1,1,1,1,2


,booking_id,show_id,movie_title,theatre_name,screen,seats_capacity,show_date,show_time
32,BK0000033,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
40,BK0000041,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
395,BK0000396,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
434,BK0000435,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
435,BK0000436,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
436,BK0000437,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
454,BK0000455,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
455,BK0000456,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
456,BK0000457,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15
457,BK0000458,SH000004,Dilli Nights,Skyline Multiplex - Andheri,Screen 4,170,2025-10-18,12:15


**What we see:** movie, theatre, screen and capacity are consistent for every
show. Exactly 20 shows have two different dates and 30 have two different
times — the same 50 corrupted rows we found in Step 4.

**Treatment:** because each affected show has several other correct bookings,
we can repair the bad row with the **modal** (most common) date or time for
that `show_id`. No show loses all of its bookings, so no show disappears.

**Why not just drop the bad rows?** Dropping them would remove valid revenue
and leave some shows with artificially low occupancy. Repairing keeps the
booking and restores the true show attribute.

## Step 9 — Occupancy consistency

Occupancy is the metric most often calculated incorrectly. It is **not**
`seats / seats_capacity` for a single booking, because one booking is only one
transaction in a show that may have dozens of bookings.

The correct calculation is:

```
occupancy(show) = SUM(seats of Confirmed bookings for that show) / seats_capacity
```

Cancelled bookings are excluded because those seats were released. We compute
it on the **deduplicated** data and then again on the raw data, so we can see
the damage duplicates do.

In [17]:
unique_df = df.drop_duplicates(subset="booking_id", keep="first")

def occupancy_table(frame):
    confirmed = frame[frame["booking_status"] == "Confirmed"]
    seats_sold = confirmed.groupby("show_id")["seats"].sum()
    capacity = frame.groupby("show_id")["seats_capacity"].first()
    return (seats_sold.reindex(capacity.index, fill_value=0) / capacity).rename("occupancy")

occ_clean = occupancy_table(unique_df)
occ_raw = occupancy_table(df)

print("Deduplicated data:")
print("  shows        :", len(occ_clean))
print("  min / mean / max:", round(occ_clean.min(), 3), round(occ_clean.mean(), 3), round(occ_clean.max(), 3))
print("  shows > 100% :", int((occ_clean > 1).sum()))

print("\nRaw data (duplicates included):")
print("  shows        :", len(occ_raw))
print("  min / mean / max:", round(occ_raw.min(), 3), round(occ_raw.mean(), 3), round(occ_raw.max(), 3))
print("  shows > 100% :", int((occ_raw > 1).sum()))

raw_confirmed = df[df["booking_status"] == "Confirmed"]
clean_confirmed = unique_df[unique_df["booking_status"] == "Confirmed"]
print("\nDuplicate impact on the KPIs:")
print("  Confirmed seats   raw / deduped:",
      f"{raw_confirmed['seats'].sum():,}", "/", f"{clean_confirmed['seats'].sum():,}")
print("  Confirmed revenue raw / deduped:",
      f"INR {raw_confirmed['total_amount'].sum():,}", "/",
      f"INR {clean_confirmed['total_amount'].sum():,}")

Deduplicated data:
  shows        : 290
  min / mean / max: 0.103 0.412 0.907
  shows > 100% : 0

Raw data (duplicates included):
  shows        : 290
  min / mean / max: 0.108 0.424 0.945
  shows > 100% : 0

Duplicate impact on the KPIs:
  Confirmed seats   raw / deduped: 23,603 / 22,920
  Confirmed revenue raw / deduped: INR 11,161,531 / INR 10,823,192


In [18]:
occupancy_summary = pd.DataFrame({
    "occupancy": occ_clean,
    "seats_capacity": unique_df.groupby("show_id")["seats_capacity"].first(),
    "confirmed_seats": unique_df[unique_df["booking_status"] == "Confirmed"]
        .groupby("show_id")["seats"].sum(),
}).fillna({"confirmed_seats": 0})

display(occupancy_summary.sort_values("occupancy", ascending=False).head(8))
display(occupancy_summary.sort_values("occupancy").head(8))

,occupancy,seats_capacity,confirmed_seats
show_id,,,
SH000045,0.906897,290,263
SH000105,0.900000,220,198
SH000221,0.896667,300,269
SH000016,0.891667,120,107
SH000135,0.883333,180,159
SH000198,0.878571,140,123
SH000231,0.878571,280,246
SH000226,0.877778,180,158


,occupancy,seats_capacity,confirmed_seats
show_id,,,
SH000289,0.103030,330,34
SH000027,0.108333,120,13
SH000251,0.118750,160,19
SH000119,0.122222,180,22
SH000263,0.125000,120,15
SH000204,0.126316,190,24
SH000113,0.127778,180,23
SH000285,0.128125,320,41


**What we see:** on the deduplicated data, occupancy stays between 0 and 1
(5.0% to 91.5%, mean 40.2%). On the raw data, the mean rises to 41.4% and
confirmed seats/revenue are inflated by about 3%, because duplicates are
counted as extra sales. No show crosses 100% in this sample — the fullest show
has 8.5% headroom — but the ratio is still overstated, and in a busier sample
the same bug would produce impossible values above 100%.

This is why we deduplicate **before** calculating ratios: a duplicate is not
an extra sale, and ratios are especially sensitive because the same row
inflates the numerator.

**Treatment:** deduplicate first, exclude `Cancelled` bookings, and store the
result as `show_occupancy_rate` in the cleaned file. The value is identical for
every booking of the same show, because it describes the show, not the booking.

## Step 10 — Customer consistency

A `customer_id` should describe one person. That person's name, gender and home
city must not change between bookings.

We check this after normalizing the city labels, because `" Bangalore "` and
`"Bengaluru"` are the same city even though the raw strings differ.

In [19]:
# Deduplicate first. A repeated row is not a second visit, so counting it
# would overstate how often each customer books.
customers_df = df.drop_duplicates(subset="booking_id", keep="first")

customer_check = customers_df.groupby("customer_id").agg(
    bookings=("booking_id", "count"),
    names=("customer_name", "nunique"),
    genders=("gender", "nunique"),
    cities=("city_norm", "nunique"),
)

print("Distinct customers:", len(customer_check))
print("Customers with >1 name  :", int((customer_check["names"] > 1).sum()))
print("Customers with >1 gender:", int((customer_check["genders"] > 1).sum()))
print("Customers with >1 city  :", int((customer_check["cities"] > 1).sum()))

print("\nBookings per customer (deduplicated):")
display(customer_check["bookings"].describe().round(2))

Distinct customers: 3499
Customers with >1 name  : 0
Customers with >1 gender: 0
Customers with >1 city  : 0

Bookings per customer (deduplicated):


count    3499.00
mean        2.86
std         2.36
min         1.00
25%         1.00
50%         2.00
75%         3.00
max        18.00
Name: bookings, dtype: float64

In [20]:
# Segments describe customers, so they must be measured AFTER deduplication.
# A duplicated row is not an extra visit: leaving duplicates in would promote
# a few one-booking customers to "Returning" and a few "Returning" to "Loyal".
def segment_table(frame):
    counts = frame.groupby("customer_id").size()
    labels = counts.map(
        lambda n: "New" if n == 1 else ("Returning" if n <= 3 else "Loyal")
    )
    table = pd.DataFrame({
        "customers": labels.value_counts(),
        "bookings": frame["customer_id"].map(labels).value_counts(),
    }).reindex(["New", "Returning", "Loyal"]).fillna(0).astype(int)
    table["customer_share_%"] = (
        table["customers"] / table["customers"].sum() * 100
    ).round(1)
    table["booking_share_%"] = (
        table["bookings"] / table["bookings"].sum() * 100
    ).round(1)
    return table

print("Deduplicated (the numbers we will use):")
display(segment_table(customers_df))

print("Raw file, for comparison (duplicates inflate the Loyal segment):")
display(segment_table(df))

Deduplicated (the numbers we will use):


,customers,bookings,customer_share_%,booking_share_%
New,1112,1112,31.8,11.1
Returning,1527,3653,43.6,36.5
Loyal,860,5235,24.6,52.4


Raw file, for comparison (duplicates inflate the Loyal segment):


,customers,bookings,customer_share_%,booking_share_%
New,1087,1087,31.1,10.6
Returning,1492,3575,42.6,34.7
Loyal,920,5638,26.3,54.7


**What we see:** every customer has exactly one name, one gender and one city
after normalization. That is what we want, and it confirms the city variants
were only spelling differences, not different people.

The segment table shows why `customer_id` matters. In v1 every booking had a
unique customer, so retention analysis was impossible. v2 has genuine repeat
bookings: Loyal customers are **25.4% of customers but 53.3% of bookings**,
while New customers are 31.3% of customers but only 10.9% of bookings. That is
a realistic pattern and gives us something meaningful to analyse later.

> The comparison table is deliberately included: on the raw file, Loyal looks
> like 26.8% of customers and 55.2% of bookings. Deduplication moves 49
> customers and 355 bookings out of that segment. The difference is small here,
> but it is the same mistake as the occupancy error in Step 9 — a ratio is only
> as trustworthy as the rows underneath it.

> The segment rule is a **snapshot** over the 12-month window: New = 1 booking,
> Returning = 2–3, Loyal = 4+. A customer who booked once in the window may
> have been a customer before it, so this is a behavioural segment, not a
> lifetime-value label.

## Step 11 — Issue register and treatment plan

Now we collect every problem we found into one table. This is the bridge to
the cleaning notebook: nothing gets changed unless it appears here with a
reason and a treatment.

The counts below are calculated from the file, not typed by hand.

In [21]:
issue_register = pd.DataFrame([
    {
        "Issue": "Duplicate booking_id rows",
        "Rows affected": int(df["booking_id"].duplicated().sum()),
        "Planned treatment": "Drop duplicates, keep first occurrence",
    },
    {
        "Issue": "Missing customer_email",
        "Rows affected": int(df["customer_email"].isna().sum()),
        "Planned treatment": "Copy from the same customer's other booking; else placeholder",
    },
    {
        "Issue": "Missing phone",
        "Rows affected": int(df["phone"].isna().sum()),
        "Planned treatment": "Copy from the same customer's other booking; else placeholder",
    },
    {
        "Issue": "Missing age_group",
        "Rows affected": int(df["age_group"].isna().sum()),
        "Planned treatment": "Re-derive from age",
    },
    {
        "Issue": "Missing promo_code",
        "Rows affected": int(df["promo_code"].isna().sum()),
        "Planned treatment": "Infer from discount rate; else NONE",
    },
    {
        "Issue": "ticket_price <= 0",
        "Rows affected": int((df["ticket_price"] <= 0).sum()),
        "Planned treatment": "Recover from total_amount formula",
    },
    {
        "Issue": "Invalid show_time",
        "Rows affected": int((~df["show_time"].isin(VALID_TIMES)).sum()),
        "Planned treatment": "Use modal show_time for the same show_id",
    },
    {
        "Issue": "show_date outside period",
        "Rows affected": int(((df["show_date_parsed"] < START) | (df["show_date_parsed"] > END)).sum()),
        "Planned treatment": "Use modal show_date for the same show_id",
    },
    {
        "Issue": "rating outside 0-10",
        "Rows affected": int(((df["rating"] < 0) | (df["rating"] > 10)).sum()),
        "Planned treatment": "Use modal rating for the same movie_title",
    },
    {
        "Issue": "age outside 18-68",
        "Rows affected": int(((df["age"] < 18) | (df["age"] > 68)).sum()),
        "Planned treatment": "Use the customer's age from another booking",
    },
    {
        "Issue": "City label variants",
        "Rows affected": int((df["city"] != df["city_norm"]).sum()),
        "Planned treatment": "Strip, title-case, map Bangalore/Banglore -> Bengaluru",
    },
])

issue_register["share_of_rows_%"] = (
    issue_register["Rows affected"] / len(df) * 100
).round(2)

display(issue_register)
print("Total issue categories:", len(issue_register))

,Issue,Rows affected,Planned treatment,share_of_rows_%
0,Duplicate booking_id rows,300,"Drop duplicates, keep first occurrence",2.91
1,Missing customer_email,62,Copy from the same customer's other booking; e...,0.60
2,Missing phone,60,Copy from the same customer's other booking; e...,0.58
3,Missing age_group,50,Re-derive from age,0.49
4,Missing promo_code,51,Infer from discount rate; else NONE,0.50
5,ticket_price <= 0,51,Recover from total_amount formula,0.50
6,Invalid show_time,31,Use modal show_time for the same show_id,0.30
7,show_date outside period,21,Use modal show_date for the same show_id,0.20
8,rating outside 0-10,10,Use modal rating for the same movie_title,0.10
9,age outside 18-68,11,Use the customer's age from another booking,0.11


Total issue categories: 11


### What cleaning will do (and will not do)

**Will do**

- Deduplicate by `booking_id` → 10,000 bookings.
- Normalize city labels.
- Repair missing values using information already in the file.
- Repair impossible prices, times, dates, ratings and ages using the
  show/customer/movie relationships.
- Recompute `total_amount` from its components.
- Derive `age_group`, `customer_type` and `show_occupancy_rate`.
- Validate every repair and write a before/after log.

**Will not do**

- Delete rows just because a value is unusual.
- Fill every blank with zero.
- Remove outliers without checking whether they are genuine.
- Invent a value when no evidence exists in the file.

### Validation targets for the cleaned file

| Check | Target |
|---|---|
| Rows | exactly 10,000 |
| Unique `booking_id` | exactly 10,000 |
| Missing values in required fields | 0 |
| Revenue-formula failures | 0 |
| Occupancy outside 0–1 | 0 |
| Invalid ages / ratings / prices | 0 |
| City labels | exactly 8 canonical cities |
| Shows with inconsistent date/time | 0 |
| Customers with inconsistent name/gender/city | 0 |

The raw file will not be modified. Cleaning writes a **new** file,
`data/clean/bookings_clean.csv`, so the original evidence is preserved.

## Practice tasks

Try these before moving to the cleaning notebook. They use only what we have
already done above.

1. **Count, don't guess.** How many rows have a `promo_code` blank **and** a
   `discount_amount` greater than zero? Why can those promos be recovered,
   while a blank promo with zero discount cannot?
2. **Duplicate impact.** Calculate confirmed revenue on the raw file and on
   the deduplicated file. What is the difference in rupees and percent?
3. **Occupancy by show.** Find the show with the highest occupancy and the
   show with the lowest. How many bookings does each have? Does the lowest one
   look like a genuine flop or a data problem?
4. **City merge.** Before normalization, what share of bookings does each
   Bengaluru spelling have? After normalization, what is the combined share?
5. **Customer segments.** Change the Loyal threshold from 4+ to 5+ bookings.
   How many customers and bookings move from Loyal to Returning? Why does the
   choice of threshold matter for a business recommendation?

Write your answers in a new markdown cell below this one. Do not change the
raw CSV.